## Data Description and Cleaning

### Job-posting data

We used Luke Barousse’s data_jobs.csv dataset which contains online job postings that are collected through Google from several job sites. Each row represents one job posting. We used the posting date, country, standardized job title and list of skills. The dataset uses a model to standardize the job titles and extracts skill names from the postings.

We kept the postings from 2023 that were labeled United States and had a standardized job title of Data Analyst, Data Scientist or Data Engineer. We then removed exact duplicate rows and excluded postings without a usable skill list. We counted each skill at most once per posting. After cleaning, we had 141,910 postings. For each role- we calculated the percentage mentioning a skill by dividing the number of postings that listed that skill by the total number of postings for that role with a usable skill list.

The dataset covers postings that appeared in the collector’s searches. Jobs advertised offline or missed by those searches are outside its coverage. The automated labels may also miss skills or assign a posting to the wrong role.

[Hugging Face dataset and documentation](https://huggingface.co/datasets/lukebarousse/data_jobs) · [Original CSV file](https://huggingface.co/datasets/lukebarousse/data_jobs/tree/main)

In [15]:
import ast
import pandas as pd

jobs = pd.read_csv("job_postings_2023_clean.csv")
courses = pd.read_csv("cornell_courses_2023_clean.csv")

# The CSV stores each skill list as text. Convert it back to a list.
jobs["job_skills"] = jobs["job_skills"].map(ast.literal_eval)

print("Job postings:", len(jobs))
print("Cornell courses:", len(courses))

Job postings: 141910
Cornell courses: 398


In [16]:
postings_per_role = jobs["job_title_short"].value_counts()
job_skills = jobs.explode("job_skills", ignore_index=True)
# Select the 20 most common skills across all three roles.
top_skills = job_skills["job_skills"].value_counts().head(20).index

skill_counts = pd.crosstab(
    job_skills["job_skills"], job_skills["job_title_short"]
).reindex(top_skills)

skill_percentages = (
    skill_counts.div(postings_per_role, axis="columns") * 100
).round(1)

skill_percentages

job_title_short,Data Analyst,Data Engineer,Data Scientist
job_skills,,,
sql,60.0,71.9,58.1
python,32.0,68.2,82.0
r,20.3,9.5,50.4
tableau,33.7,15.1,26.9
excel,47.9,6.0,12.8
aws,5.8,45.0,19.9
spark,2.4,33.7,19.1
azure,5.0,33.8,13.3
power bi,20.2,10.4,11.0


In [17]:
import re

# Use more specific patterns where a skill name can mean something else.
special_patterns = {
    "r": r"\bR\b",
    "excel": r"\bExcel\b",
    "word": r"(?i)\b(?:Microsoft|MS)\s+Word\b",
    "spark": r"(?i)\b(?:Apache\s+Spark|PySpark)\b|\bSpark\b",
    "azure": r"(?i)\bMicrosoft\s+Azure\b|\bAzure\b",
    "oracle": r"\bOracle\b",
    "snowflake": r"\bSnowflake\b",
"sql": r"(?i)\bSQL\b(?!\s+Server\b|[-–—]\s*like\b)",}

descriptions = courses["description"].fillna("")
cornell_counts = {}

for skill in top_skills:
    pattern = special_patterns.get(skill, rf"(?i)\b{re.escape(skill)}\b")
    cornell_counts[skill] = descriptions.str.contains(pattern).sum()

comparison = skill_percentages.copy()
comparison["Cornell course count"] = pd.Series(cornell_counts)
comparison["Cornell courses (%)"] = (
    comparison["Cornell course count"] / len(courses) * 100
).round(1)

comparison

job_title_short,Data Analyst,Data Engineer,Data Scientist,Cornell course count,Cornell courses (%)
job_skills,,,,,
sql,60.0,71.9,58.1,2,0.5
python,32.0,68.2,82.0,16,4.0
r,20.3,9.5,50.4,26,6.5
tableau,33.7,15.1,26.9,0,0.0
excel,47.9,6.0,12.8,3,0.8
aws,5.8,45.0,19.9,0,0.0
spark,2.4,33.7,19.1,1,0.3
azure,5.0,33.8,13.3,0,0.0
power bi,20.2,10.4,11.0,0,0.0


### Exploratory comparison of job postings and course descriptions

SQL appeared in at least 58% of postings with usable skill lists in each of the three roles. Other skills varied more by role. Excel appeared in 47.9% of data analyst postings, compared with 6.0% of data engineer postings and 12.8% of data scientist postings. Python appeared in 82.0% of data scientist postings, compared with 68.2% of data engineer postings and 32.0% of data analyst postings.

Among the 398 Cornell courses we examined, 26 descriptions mentioned R (6.5%), 16 mentioned Python (4.0%), and 2 mentioned SQL (0.5%). The two sources describe different things. Job postings list skills employers ask for, while short course descriptions may leave out tools students use in class. These results tell us which skills were explicitly mentioned in each source.

In [18]:
# Show the words around each R match so we can check what it means.
for course_id, description in zip(courses["crseId"], descriptions):
    for match in re.finditer(special_patterns["r"], description):
        start = max(0, match.start() - 60)
        end = match.end() + 60
        snippet = description[start:end].replace("\n", " ")
        print(course_id, snippet)

352214 me securities, and resampling techniques. Covers the use of R for statistical calculations, simulation, and optimization.
352376 odeling and parallel computing. Programming will be done in R.
358529 ming (VBA), and programming in a scripting language (either R or Python). Operations research methods include inventory m
358808 les of statistical programming (such as data frames, Python/R packages, reproducible workflows, and version control), uni
359479  bayesian statistics. Students are instructed in the use of R software; prior knowledge of R is helpful but not required.
359479 are instructed in the use of R software; prior knowledge of R is helpful but not required. This course is intended for M.
360952  to linear modeling. Applied analysis is carried out in the R statistical computing environment.
364083 rs. Homework assignments will involve computation using the R statistical package.
364513 methods. Some assignments may involve computation using the R programming language.


In [19]:
# Show the context around the remaining skill matches.
for skill in top_skills:
    if skill == "r":
        continue

    pattern = special_patterns.get(skill, rf"(?i)\b{re.escape(skill)}\b")
    for course_id, description in zip(courses["crseId"], descriptions):
        for match in re.finditer(pattern, description):
            start = max(0, match.start() - 60)
            end = match.end() + 60
            snippet = description[start:end].replace("\n", " ")
            print(skill, course_id, snippet)

sql 358529  science. Software tools include structured query language (SQL), geographical information systems (GIS), Excel and Visual 
sql 358551 atabases, learning about querying via the database language SQL. Through a succession of projects, students learn how to ap
python 352187 eeded in practice; in particular, modeling in simulation in Python, as well as commercial discrete-event simulation languages.
python 353363 eded in practice; in particular, modeling and simulation in Python, as well as commercial discrete-event simulation languages.
python 358525 Programming and problem solving using Python. Emphasizes the systematic development of algorithms and pr
python 358526 Programming and problem solving using Python. Emphasizes principles of software development, style, and 
python 358529 (VBA), and programming in a scripting language (either R or Python). Operations research methods include inventory management,
python 358531 titative trading in Finance. The main computational too